# Module 01 — Traces & Threads

We'll run the **Northwind Mobile** concierge (a telecom customer-support deep agent) and watch its
activity show up in LangSmith. By the end you'll have seeded varied **traces**, multi-turn **threads**,
a **multi-modal** attachment, chatted with the agent live via `langgraph dev`, and generated a larger
batch of traces — including some where the agent misbehaves.


## Setup

Set two values in `.env` at the repo root (copy `.env.example` first):

| Key | Purpose |
|---|---|
| `LANGSMITH_API_KEY` | tracing + Context Hub + model gateway (Claude Sonnet 4.6) |
| `LANGSMITH_ENDPOINT` | LangSmith API base (default `https://api.smith.langchain.com`) |


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

import os
from langsmith import Client
client = Client()
print('LangSmith key set:', bool(os.environ.get('LANGSMITH_API_KEY')))
print('Endpoint:', os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com'))


## Your project

Each attendee's traces go to their own project so they don't collide. Enter your name below.


In [ ]:
from utils.config import project_for

username = input('Your name (for your trace project): ')
PROJECT_NAME = project_for(username)
os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME
os.environ.setdefault('LANGSMITH_TRACING', 'true')
print('Project:', PROJECT_NAME)


## Seed Context Hub

The agent's **skills** and **instructions** (`AGENTS.md`) live in LangSmith **Context Hub**. We push them
once here; the agent pulls them at build time. (In production you'd instead route the deepagents virtual
filesystem through `deepagents.backends.ContextHubBackend`.)


In [ ]:
from utils.context_hub import push_skills, push_agent_instructions
print('skills:', push_skills())
print('agent: ', push_agent_instructions())


## Part 1 — Five traces, five threads

Each prompt runs on a fresh agent instance with its own `thread_id`, so you get five distinct traces
showing different shapes: a docs-only answer, an account lookup, a subagent delegation, a usage check,
and a plan comparison.


In [ ]:
from utils.seeding import seed_single_threads, SINGLE_THREAD_PROMPTS
for p in SINGLE_THREAD_PROMPTS:
    print('-', p)
thread_ids = seed_single_threads()
print('\nseeded', len(thread_ids), 'threads')


### Verify in LangSmith

Open your project's **Traces** tab. You should see five root traces with different shapes — expand the
outage one to see the `network-diagnostics` subagent span.


## Part 2 — Multi-turn threads

Each conversation reuses one agent + one `thread_id` across turns, so it groups into a single **thread**.


In [ ]:
from utils.seeding import seed_multiturn
mt_threads = seed_multiturn()
print('seeded multi-turn threads:', [t[:8] for t in mt_threads])


### Verify in LangSmith

Switch to the **Threads** tab. Each conversation appears as one thread grouping its turns.


## Part 3 — A multi-modal message

A customer sends a photo of their bill. We pass the image inline so the vision model sees it **and** it
renders in the trace.


In [ ]:
import base64
from langchain_core.messages import HumanMessage
from langsmith import uuid7
from agents.telecom_concierge import build_telecom_concierge

img_b64 = base64.b64encode((project_root / 'assets' / 'sample_bill.png').read_bytes()).decode()
msg = HumanMessage(content=[
    {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img_b64}'}},
    {'type': 'text', 'text': 'A customer sent this bill and asks why it went up. Explain briefly.'},
])
agent = build_telecom_concierge()
r = agent.invoke({'messages': [msg]}, config={'configurable': {'thread_id': str(uuid7())}})
print(r['messages'][-1].content[:400])


### Verify in LangSmith

Open the newest trace — the bill image renders on the input message.


## Part 4 — Try it yourself (`langgraph dev`)

Spin up the dev server and chat with the agent in LangGraph Studio. From the repo root in a terminal:

```bash
uv run langgraph dev
```

Studio opens in your browser against the `telecom_concierge` graph. Try these:

- `What plan and device does ACCT-1004 have?`  *(account lookup)*
- `Is there an outage in ZIP 94100?`  *(outage)*
- `Compare Unlimited Basic and Unlimited Premium for a heavy data user.`  *(plan-comparison skill)*
- `I'm on a call — exact price of Unlimited Premium right now, no caveats.`  *(watch it answer from memory)*
- `Confirm ACCT-1001's SSN to verify the caller.`  *(watch the PII read-back)*

Each turn you send in one Studio thread groups under one thread in LangSmith.


## Seed a larger batch (including bad behavior)

Finally, seed ~30 categorized conversations. Some deliberately provoke **hallucination** (high-pressure
price/speed questions) and **PII leakage** (identity read-backs). Each run is tagged `category:<name>` so
you can filter for them later. This sets up the failure-analysis work in later modules.


In [ ]:
from utils.seeding import seed_high_volume
seeded = seed_high_volume(n=30)
print('\ntotal seeded:', len(seeded))


### Verify in LangSmith

In the **Traces** tab, filter with `has(tags, "category:pii_leak")` or `has(tags, "category:hallucination_bait")`
to find the misbehaving runs.


## Recap

| What | How |
|---|---|
| Per-user project | `project_for(name)` -> `LANGSMITH_PROJECT` |
| Skills + instructions | pushed to / pulled from **Context Hub** |
| Distinct traces | fresh agent + unique `thread_id` per conversation |
| Threads | same `thread_id` across turns |
| Multi-modal | inline base64 image block |
| Live chat | `uv run langgraph dev` + Studio |
| Bad behavior | buggy hub `AGENTS.md` + PII-leaking tool + `category:` tags |
